In [39]:
!pip install kfp-kubernetes==1.4.0 kserve==0.15.2 kubernetes==26.1.0

In [40]:
import kfp
from kfp import dsl
from kfp import kubernetes
from kfp import local

# TIP: you may need to authenticate with the KFP instance
# local.init(runner=local.SubprocessRunner())
kfp_client = kfp.Client()

In [41]:
current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()
namespace_cur = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.metadata.namespace}'").read()
print(namespace_cur)
print(current_sc)

geun-tak-roh-2e590eb8
gl4f-filesystem


## Check Container file-system architecture & workdir

In [42]:
@dsl.component()
def download_dataset(download_url: str, output_file: str, target_path: str, dataset_pvc_name: str) -> str:
    import os
    import subprocess
    
    result = subprocess.run(['curl','-L','-o', output_file, download_url],capture_output=True, text=True)
    if result.returncode == 0:
        if os.path.exists(output_file):
            print(f"Successfully downloaded {output_file}")
            print(f"File size: {os.path.getsize(output_file)} bytes")
        else:
            print("Download completed but file not found")
    else:
        print(f"Curl failed with return code {result.returncode}")
        print(f"Error: {result.stderr}")
    
    subprocess.run(['pwd'])
    subprocess.run(['ls','-al'])
    # unzip into the PVC 
    subprocess.run(['unzip',output_file,'-d', target_path])
    return dataset_pvc_name

In [45]:
@dsl.pipeline(
    name="down-dataset-pipe"
)
def download_dataset_pipeline(download_url: str, output_file: str, dataset_pvc_name: str, dataset_pvc_size: str,current_sc: str) -> str:
    pvc1 = kubernetes.CreatePVC(
        # can also use pvc_name instead of pvc_name_suffix to use a pre-existing PVC
        # pvc_name=dataset_pvc_name,
        pvc_name_suffix=dataset_pvc_name,
        access_modes=['ReadWriteMany'],
        size=dataset_pvc_size,
        storage_class_name=current_sc, # gl4fs-system for PCAI
    )
    # write to the PVC
    target_path = '/data'
    task1 = download_dataset(download_url=download_url,output_file=output_file,target_path=target_path,dataset_pvc_name=pvc1.outputs['name'])
    kubernetes.mount_pvc(
        task1,
        pvc_name=pvc1.outputs['name'],
        # pvc_name='roboflow-license-plate-datasets',
        mount_path=target_path,
    )
    return task1.output

In [ ]:
download_url = "" # Enter your url from roboflow
output_file = "/data/roboflow.zip"
dataset_pvc_name = "roboflow-lp-datasets"
dataset_pvc_size = '5Gi'
current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()

kfp_client.create_run_from_pipeline_func(
    download_dataset_pipeline,
    arguments={
        'download_url': download_url,
        'output_file': output_file,
        'dataset_pvc_name': dataset_pvc_name,
        'dataset_pvc_size': dataset_pvc_size,
        'current_sc': current_sc
    },
    experiment_name="test-rhgt-exp",
    # enable_caching=False # failed: failed to create PVC and publish execution createpvc: failed to create cache entrty for create pvc: failed to create task: rpc error: code = InvalidArgument desc = Failed to create a new task due to validation error: Invalid input error: Invalid task: must specify FingerPrint
    # enable_caching=True
)

RunPipelineResult(run_id=96284ba2-26d0-4b04-9735-633ee43759c8)

In [47]:
from kfp import compiler, dsl

compiler.Compiler().compile(download_dataset_pipeline, package_path='download_dataset_pipeline.yaml')